# Chatbot: Persistence (memory with a checkpointer)

Same graph as `simple_chatbot.ipynb`, but compiled with a **checkpointer**. It saves the state
after every run, per `thread_id`, so the next invoke on the same thread continues the conversation.

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState):
    response = model.invoke(state['messages'])
    return {'messages': [response]}

In [3]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

checkpointer = InMemorySaver()  # saves state in RAM (lost when the kernel restarts)
chatbot = graph.compile(checkpointer=checkpointer)

In [4]:
# thread_id = which conversation this message belongs to
config = {'configurable': {'thread_id': '1'}}

chatbot.invoke({'messages': [HumanMessage(content='Hi, my name is Priyansh')]}, config=config)
result = chatbot.invoke({'messages': [HumanMessage(content='What is my name?')]}, config=config)
print(result['messages'][-1].content)

Your name is **Priyansh**.


In [5]:
# A different thread is a separate conversation, so it knows nothing about thread 1
config2 = {'configurable': {'thread_id': '2'}}

result = chatbot.invoke({'messages': [HumanMessage(content='What is my name?')]}, config=config2)
print(result['messages'][-1].content)

I don’t actually have that information. Could you tell me your name?


In [6]:
# The saved state of a thread can be read back at any time
state = chatbot.get_state(config)
for message in state.values['messages']:
    print(f'{message.type}: {message.content}')

human: Hi, my name is Priyansh
ai: Hello Priyansh! 👋 How can I assist you today?
human: What is my name?
ai: Your name is **Priyansh**.


In [7]:
# A checkpoint is saved after every step, so the whole history is kept
for snapshot in chatbot.get_state_history(config):
    print(len(snapshot.values.get('messages', [])), 'messages, next:', snapshot.next)

4 messages, next: ()
3 messages, next: ('chat_node',)
2 messages, next: ('__start__',)
2 messages, next: ()
1 messages, next: ('chat_node',)
0 messages, next: ('__start__',)


In [ ]:
# Chat in a loop with memory. Type 'exit' to stop.
config = {'configurable': {'thread_id': 'chat'}}

while True:
    user_message = input('You: ')
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    result = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI:', result['messages'][-1].content)